# 06. Delay-SNN 分支的量化（训练后量化，不需要重新训练）

Context（notebook 02）和 Hybrid（notebook 04）都用了 QAT——把量化误差引入训练循环，
重新微调几十个 epoch才能找回精度。这个 notebook 展示一个**不同的、更简单的路线**：
**PTQ（Post-Training Quantization，训练后量化）**——直接对 notebook 05 训练好的
FP32 权重做量化，不做任何再训练。这不是偷懒，是因为这个分支的设计本身就更容易
量化：固定的 decay=0.9（不是学出来的、可能对精度敏感的值）、没有 LayerNorm/GELU/
除法这些数据依赖的算子、网络本身也更小更浅。教程 05 章提过"PTQ vs QAT怎么选"，
这里是一个真实的"PTQ 就够用"的案例，可以和 Context/Hybrid 的"必须上 QAT"案例对比着看。

这个分支还有一点特殊：它是**唯一一个真正综合、跑在了 Nexys4 DDR 开发板上的分支**
（另外两个分支目前只做到了软件端的定点验证，见教程 08 章的"什么能验证、什么不能"）。
这个 notebook 复现的定点算法，就是 `optional local FPGA implementation project/`
项目里实际拿去做 RTL 仿真、综合、烧录的那一套。

In [ ]:
import sys
from pathlib import Path

import numpy as np
import torch

DELAY_ROOT = Path("training/semg_snn_fpga_reproduction")
sys.path.insert(0, str(DELAY_ROOT))
from model import PaperSNNWithDelays

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

NB_RUNS = DELAY_ROOT / "runs_notebook"
FP32_DELAY_CHECKPOINT = NB_RUNS / "delay62_finetune_nb" / "best.pt"
if not FP32_DELAY_CHECKPOINT.exists():
    FP32_DELAY_CHECKPOINT = DELAY_ROOT / "runs" / "delay62_finetune" / "best.pt"
print("FP32 checkpoint:", FP32_DELAY_CHECKPOINT)

state = torch.load(FP32_DELAY_CHECKPOINT, map_location=device, weights_only=False)["model"]
fp32_model = PaperSNNWithDelays(decay=0.9, threshold=1.0, max_delay=62, initial_delay=1.0).to(device)
fp32_model.load_state_dict(state)
fp32_model.eval()

## 1. 量化方案（比 Context/Hybrid 简单得多）

来自真实项目 `semg_snn_nexys4ddr_vivado/scripts/export_fixed_point.py` 的方案：

| 对象 | 量化方式 |
|---|---|
| 权重 | 逐层(每个 Linear 一个) 对称 INT8,`scale=max\|w\|/127` |
| 轴突延迟 | 四舍五入到最近的整数时间步(0~62)——延迟本来就是"第几个时间步",天然离散 |
| 膜电位 | Q8 定点,**用该层权重的 scale 作单位**(不是像 Context/Hybrid 那样用统一的 Q8.8) |
| 阈值 | `threshold_q = round(256 / scale)`,把浮点阈值 1.0 换算成同一层的 Q8 整数单位 |
| 衰减 beta | **固定为 230/256 ≈ 0.898**(不是学出来的,直接取一个接近训练用 decay=0.9 的定点值) |
| 衰减的舍入 | 有符号"最近偶数舍入"(banker's rounding),不是简单截断,减少系统性偏差 |
| 复位 | **硬复位**:发放后膜电位直接清零(比 Context/Hybrid 的减法软复位更省资源) |

关键的资源亮点：**这里输入永远是二值脉冲(0/1)，所以"权重×输入"这个乘法在硬件里
根本不需要真正的乘法器——发放的那些输入通道对应的权重直接累加即可**（`weight_i64
@ events[time_index]` 在 numpy 里看起来是矩阵乘法，但输入是 boolean，语义上等价于
"挑出发放的那些权重求和"）。这正是教程 08 章提到过、但 Context/Hybrid 还没有真正
在硬件层面利用的"脉冲门控加法代替乘法"优化——Delay-SNN 分支是本课程里唯一一个
真正做到这一点的例子。

In [ ]:
def round_shift_signed(values: np.ndarray, shift: int) -> np.ndarray:
    """有符号最近偶数舍入(RNE),再右移。对应 RTL 里 v*230 之后的整数除法。"""
    source = np.asarray(values, dtype=np.int64)
    magnitude = np.abs(source)
    quotient = magnitude >> shift
    remainder = magnitude & ((1 << shift) - 1)
    half = 1 << (shift - 1)
    increment = (remainder > half) | ((remainder == half) & ((quotient & 1) != 0))
    rounded = quotient + increment.astype(np.int64)
    return np.where(source < 0, -rounded, rounded)


def quantize_weights(state, weight_keys):
    quantized, scales, thresholds = [], [], []
    for key in weight_keys:
        weight = state[key].detach().cpu().numpy().astype(np.float64)
        scale = float(np.max(np.abs(weight)) / 127.0)
        integer = np.rint(weight / scale).clip(-127, 127).astype(np.int8)
        quantized.append(integer)
        scales.append(scale)
        thresholds.append(int(np.rint((1 << 8) / scale)))  # threshold=1.0 换算成该层 Q8 单位
    return quantized, scales, thresholds


def quantize_delays(state, delay_keys, max_delay=62):
    output = []
    for key in delay_keys:
        logits = state[key].detach().cpu()
        delay = max_delay * torch.sigmoid(logits)
        output.append(torch.round(delay).clamp(0, max_delay).to(torch.uint8).numpy())
    return output

WEIGHT_KEYS = tuple(f"layers.{i}.synapse.weight" for i in range(4))
DELAY_KEYS = tuple(f"delays.{i}.delay_logits" for i in range(3))

quantized_weights, weight_scales, thresholds_q = quantize_weights(state, WEIGHT_KEYS)
integer_delays = quantize_delays(state, DELAY_KEYS)

for index, (w, s, t) in enumerate(zip(quantized_weights, weight_scales, thresholds_q)):
    print(f"layer {index}: weight shape={w.shape}  scale={s:.5f}  threshold_q={t}  "
          f"code range=[{w.min()},{w.max()}]")
for index, d in enumerate(integer_delays):
    print(f"delay {index}: range=[{d.min()},{d.max()}]  mean={d.mean():.2f}")

## 2. 纯整数前向推理（和 RTL 位对位一致）

In [ ]:
INT32_MIN, INT32_MAX = -(1 << 31), (1 << 31) - 1
DECAY_NUMERATOR, FRACTION_BITS = 230, 8

def fixed_layer(events: np.ndarray, weight: np.ndarray, threshold_q: int) -> np.ndarray:
    """一层 Dense LIF 的整数前向。events: [time, in_features] 二值。"""
    time_steps = events.shape[0]
    membrane = np.zeros(weight.shape[0], dtype=np.int64)
    output = np.zeros((time_steps, weight.shape[0]), dtype=np.bool_)
    weight_i64 = weight.astype(np.int64)
    for t in range(time_steps):
        # events 是布尔值,这一步等价于"把发放的输入通道对应的权重加起来",不需要乘法器
        synaptic = weight_i64 @ events[t].astype(np.int64)
        decayed = round_shift_signed(membrane * DECAY_NUMERATOR, FRACTION_BITS)
        pre_reset = np.clip(decayed + (synaptic << FRACTION_BITS), INT32_MIN, INT32_MAX)
        spikes = pre_reset >= threshold_q
        output[t] = spikes
        membrane = np.where(spikes, 0, pre_reset)  # 硬复位:发放后清零
    return output


def apply_delays(events: np.ndarray, delays: np.ndarray) -> np.ndarray:
    delayed = np.zeros_like(events)
    for channel, amount in enumerate(delays):
        amount = int(amount)
        if amount == 0:
            delayed[:, channel] = events[:, channel]
        elif amount < len(events):
            delayed[amount:, channel] = events[:-amount, channel]
    return delayed


def fixed_inference(events: np.ndarray, weights, delays, thresholds) -> tuple[int, np.ndarray]:
    """events: [100, 96] 二值输入 -> (预测类别, 每类输出脉冲计数)。"""
    current = np.asarray(events, dtype=np.bool_)
    for index, (weight, threshold) in enumerate(zip(weights, thresholds)):
        current = fixed_layer(current, weight, threshold)
        if index < len(delays):
            current = apply_delays(current, delays[index])
    counts = current.sum(axis=0)
    return int(np.argmax(counts)), counts

## 3. 三档对比：FP32 / 仅权重 PTQ / 完整定点（RTL 等效）

真实项目在 `evaluate_checkpoint.py` 里还试过一个更粗糙的中间档：只把权重做
per-tensor fake-INT8（膜电位、延迟仍然是浮点），不改变其它任何东西——这是最简单的
PTQ 形式，几乎零工程成本。拿三档做对比，能看到"量化改动越多，精度损失越大，
但即使是最激进的完整定点版本，损失也只有不到 1 个百分点"。

In [ ]:
import copy

def quantize_weights_int8_simple(model):
    """最简单的 PTQ:对每个参数张量整体做对称 fake-INT8(评估脚本里的原始版本)。"""
    quantized = copy.deepcopy(model)
    with torch.no_grad():
        for parameter in quantized.parameters():
            maximum = parameter.abs().max()
            if maximum == 0:
                continue
            scale = maximum / 127.0
            parameter.copy_(torch.round(parameter / scale).clamp(-127, 127) * scale)
    return quantized


@torch.no_grad()
def evaluate_fp32_or_fake(model, x, y, batch_size=256):
    from model import spike_rate_loss
    predictions = []
    for start in range(0, len(x), batch_size):
        batch = torch.from_numpy(x[start:start+batch_size].astype(np.float32)).to(device)
        output, _ = model(batch)
        predictions.append(output.sum(dim=1).argmax(dim=1).cpu().numpy())
    predictions = np.concatenate(predictions)
    return {
        "accuracy": float((predictions == y).mean()),
        "correct": int((predictions == y).sum()),
        "total": len(y),
    }


test_data = np.load(DELAY_ROOT / "data" / "processed" / "test.npz")
x_test, y_test = test_data["x"], test_data["y"].astype(np.int64)

fp32_metrics = evaluate_fp32_or_fake(fp32_model, x_test, y_test)
fake_int8_model = quantize_weights_int8_simple(fp32_model).to(device)
fake_int8_metrics = evaluate_fp32_or_fake(fake_int8_model, x_test, y_test)

# 完整定点(逐样本跑,数据量大所以只在验证集抽样跑,示范用;全量在下一格)
sample_idx = np.random.RandomState(0).choice(len(x_test), size=1000, replace=False)
fixed_predictions = np.array([
    fixed_inference(x_test[i], quantized_weights, integer_delays, thresholds_q)[0]
    for i in sample_idx
])
fixed_sample_accuracy = float((fixed_predictions == y_test[sample_idx]).mean())

print(f"{'':28s}{'accuracy':>10s}")
print(f"{'FP32':28s}{fp32_metrics['accuracy']:10.4f}")
print(f"{'仅权重 fake-INT8 (PTQ)':28s}{fake_int8_metrics['accuracy']:10.4f}")
print(f"{'完整定点(1000 样本抽样)':28s}{fixed_sample_accuracy:10.4f}")
print("\n参考值(真实项目全量 11,276 样本测试集):")
print("  FP32:              accuracy=0.8393  macro_f1=0.6561")
print("  仅权重 fake-INT8:   accuracy=0.8349  macro_f1=0.6423")
print("  完整定点(RTL 等效): accuracy=0.8307  macro_f1=0.6317  gesture_accuracy=0.5595")

可以看到一个清晰的梯度：**改动越多，精度损失越大，但最激进的版本损失也不到
1 个百分点**（83.93% -> 83.07%）。这里量化确实付出了一点精度代价——和 Context/Hybrid
"QAT 后几乎不掉点甚至反超"不一样，这个分支没有重新训练的机会去补偿量化误差。
这个代价被认为是可以接受的，因为：(a) 三分支融合时这个分支的权重通常比较小
(真实项目里融合权重是 0.1~0.3 左右,详见 [07](07_fusion_fp32.ipynb)/[08](08_fusion_hw_qat.ipynb)
的融合结果)，(b) 换来的是一个不需要重新训练、不需要 QAT 基础设施、几十行代码就能跑通
的量化流程——对于精度不那么敏感的分支，这种"简单但有代价"的路线是合理的工程取舍。

## 4. 导出黄金测试向量

和其它分支一样，选每个类别一个代表样本，记录输入、每层脉冲发放率、最终预测和输出
计数，供硬件仿真比对。

In [ ]:
golden_vectors = []
for class_id in range(13):
    row = int(np.flatnonzero(y_test == class_id)[0])
    prediction, counts = fixed_inference(x_test[row], quantized_weights, integer_delays, thresholds_q)
    golden_vectors.append({
        "sample_index": row,
        "truth": class_id,
        "prediction": prediction,
        "correct": prediction == class_id,
        "output_counts": counts.tolist(),
    })

correct_count = sum(v["correct"] for v in golden_vectors)
print(f"{correct_count}/13 个类别代表样本预测正确")
for v in golden_vectors[:5]:
    print(v)

## 5. 这个分支已经量到了什么程度：真实硬件数字

前面 07/08 章反复强调"LUT/DSP 的具体数字需要真正跑 Vitis HLS/Vivado才能给出"——
但这个 Delay-SNN 分支是个例外：它已经完整走完了综合、实现、烧录流程,在真实
Nexys4 DDR 开发板上验证过（`semg_snn_nexys4ddr_vivado` 项目），已知的真实硬件数字是：

| 资源 | 占用 |
|---|---|
| LUT | 7.6% |
| BRAM | 7 块 |
| DSP | 0（前面提到的"脉冲门控加法代替乘法",这里是真正在硬件层面兑现了,连一个 DSP 硬核乘法器都不需要）|
| 时钟频率 | 100 MHz |

这组数字不是估算出来的,是真实综合报告里的数字——可以拿它和教程 08 章"只能定性推理"
的 Context/Hybrid LUT/DSP 讨论做对比：**同一个课程里,一个分支的资源数字是测量出来的
事实,另外两个分支的资源数字是有严格边界声明的估算**,这个区别本身就是本课程反复强调的
"诚实报告"方法论的一个具体示范。

## 下一步

三个分支的训练+量化都完成了。打开 [07_fusion_fp32.ipynb](07_fusion_fp32.ipynb) 和
[08_fusion_hw_qat.ipynb](08_fusion_hw_qat.ipynb)，把它们融合起来，复现 91.10%/91.11%
的最终结果。